# Addendum revisi: analisis tambahan untuk tanggapan reviewer (JTK)
Notebook ini **tidak mengulang** seluruh eksperimen. Notebook memakai fungsi yang sama persis dengan `hepatitis_reanalisis.ipynb` (disalin dari sana), memakai pembagian lipatan yang sama (seed 42), lalu menjalankan analisis yang diminta reviewer:

| Bagian | Komentar reviewer | Perlu menjalankan model? | Perkiraan waktu |
|---|---|---|---|
| A | *Duplicates*, *Imputation of categorical features* | Tidak | detik |
| B | *Pooled evidence for genuine leakage* | Tidak (memakai `results/hasil_semua_mentah.csv`) | detik |
| C | *Feature selection with fixed k = 8* (k = 3 dan k = 5) | Ya | ±30 menit |
| D | *Decomposition caveat* (V2b dengan 80% titik sintetis) | Ya | ±10 menit |

**Cara menjalankan:** simpan di folder utama repositori (sejajar dengan folder `data/` dan `results/`), lalu jalankan dari atas ke bawah dengan lingkungan yang sama seperti eksperimen utama. Semua hasil disimpan di `results/` dengan awalan `revisi_`.

## Bagian 0. Fungsi dan konfigurasi (disalin dari notebook utama, tanpa perubahan logika)

In [1]:
# ===== SATU-SATUNYA PENGATURAN YANG MUNGKIN PERLU DIUBAH =====
# Isi dengan lokasi folder 'hepatitis-leakage-reanalysis' di komputer masing-masing.
# Jika notebook dibuka langsung dari folder itu, pengaturan ini tidak dipakai (folder terdeteksi otomatis).
FOLDER_REPO = r"F:\ARTIKEL RAMEAN 2 -HEPATIITIS\hepatitis-leakage-reanalysis"
NAMA_PELAKSANA = "Ade"      # tulis nama yang menjalankan, misalnya "Ade" atau "Ziad"; dicatat di revisi_konfigurasi.json

import os
def _folder_sah(f):
    return all(os.path.exists(os.path.join(f, *b)) for b in
               [("data", "hcv", "hcvdat0.csv"), ("data", "hepatitis", "hepatitis.data"), ("results", "hasil_semua_mentah.csv")])
for _kandidat in (os.getcwd(), FOLDER_REPO):
    if _folder_sah(_kandidat):
        os.chdir(_kandidat)
        break
else:
    raise FileNotFoundError("Folder repositori tidak ditemukan. Isi FOLDER_REPO dengan lokasi folder "
                            "'hepatitis-leakage-reanalysis' (yang berisi folder data dan results).")
print("Folder kerja:", os.getcwd(), "| pelaksana:", NAMA_PELAKSANA)

Folder kerja: F:\ARTIKEL RAMEAN 2 -HEPATIITIS\hepatitis-leakage-reanalysis | pelaksana: Ade


In [2]:
# Instal sekali (hapus tanda # jika perlu), lalu restart kernel:
# !pip install pandas numpy scikit-learn imbalanced-learn xgboost scipy matplotlib

import os, sys, time, platform, hashlib, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from importlib.metadata import version, PackageNotFoundError

from sklearn.base import clone
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     GridSearchCV, ParameterGrid, cross_validate)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, precision_score, get_scorer
from joblib import Parallel, delayed
from imblearn.pipeline import Pipeline          # Pipeline yang mendukung SMOTE di dalam lipatan
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ---------------- KONFIGURASI (ubah di sini saja) ----------------
FAST_MODE   = False      # True = uji coba cepat (ulangan sedikit, pohon sedikit). False = hasil final.
SEED        = 42         # seed utama; dicatat di laporan
N_SPLITS    = 5          # k pada stratified k-fold
N_REPEATS   = 2 if FAST_MODE else 10
INNER_FOLDS = 3          # validasi dalam untuk tuning
K_FEATURES  = 8          # jumlah fitur pada SelectKBest
N_TREES     = 50 if FAST_MODE else 200
N_BOOT      = 500 if FAST_MODE else 5000
DATA_DIR    = "data"
OUT_DIR     = "results"
MODELS      = ["NB", "DT", "KNN", "RF", "GB", "XGB"]
SENS_MODELS = ["NB", "DT", "GB"] if FAST_MODE else MODELS
os.makedirs(OUT_DIR, exist_ok=True)

SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",            # = sensitivitas pada kelas positif
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",   # PR-AUC
}
METRICS = list(SCORING)
print("Python", platform.python_version(), "| FAST_MODE =", FAST_MODE,
      "| N_REPEATS =", N_REPEATS, "| tanggal:", datetime.date.today())

Python 3.11.8 | FAST_MODE = False | N_REPEATS = 10 | tanggal: 2026-11-05


In [3]:
print("Folder kerja notebook:", os.getcwd())
TANGGAL_AKSES = "2026-10-08"   # tanggal unduh dari UCI

# Path relatif terhadap folder utama repositori
FILES = {"HCV": os.path.join("data", "hcv", "hcvdat0.csv"),
         "Hepatitis": os.path.join("data", "hepatitis", "hepatitis.data")}
for v in FILES.values():
    assert os.path.exists(v), f"File tidak ditemukan: {v}. Jalankan notebook dari folder utama repositori."

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""):
            h.update(blok)
    return h.hexdigest()

cek = pd.DataFrame([{"dataset": k, "file": v, "ukuran_byte": os.path.getsize(v),
                     "sha256": sha256(v), "tanggal_akses": TANGGAL_AKSES}
                    for k, v in FILES.items()])
cek.to_csv(os.path.join(OUT_DIR, "checksum_data.csv"), index=False)
cek

# Data harus identik dengan eksperimen utama, karena hasil addendum dipasangkan dengan hasil lama.
SHA_ASLI = {"HCV": "1b62c8453b7098b695c549b19cfe070458ce6b909f0e990b6cfce7ab3c44c273",
            "Hepatitis": "9971f9d18abb4fc9dd6749eefb0f864b36fa7bdd59f54188c0b148a5c28214bc"}
for _, r in cek.iterrows():
    if r.sha256 != SHA_ASLI[r.dataset]:
        raise ValueError(
            f"File {r.dataset} ({os.path.abspath(r.file)}) BERBEDA dari data eksperimen utama.\n"
            "Kemungkinan file pernah dibuka lalu disimpan ulang dengan Excel (pemisah berubah menjadi ';'), "
            "atau notebook dijalankan dari folder lain.\n"
            "Solusi: jalankan notebook dari folder 'hepatitis-leakage-reanalysis' dan pakai file data asli dari repositori GitHub/Zenodo.")
print("Checksum data sesuai dengan eksperimen utama.")


Folder kerja notebook: F:\ARTIKEL RAMEAN 2 -HEPATIITIS\hepatitis-leakage-reanalysis
Checksum data sesuai dengan eksperimen utama.


In [4]:
HEP_COLS = ["Class", "AGE", "SEX", "STEROID", "ANTIVIRALS", "FATIGUE", "MALAISE", "ANOREXIA",
            "LIVER_BIG", "LIVER_FIRM", "SPLEEN_PALPABLE", "SPIDERS", "ASCITES", "VARICES",
            "BILIRUBIN", "ALK_PHOSPHATE", "SGOT", "ALBUMIN", "PROTIME", "HISTOLOGY"]

raw_hcv = pd.read_csv(FILES["HCV"])
raw_hcv = raw_hcv.rename(columns={raw_hcv.columns[0]: "X"})          # kolom ID tanpa nama
raw_hep = pd.read_csv(FILES["Hepatitis"],
                      header=None, names=HEP_COLS, na_values="?")

def periksa(df, nama, kolom_target):
    print("=" * 70)
    print(nama, "| dimensi:", df.shape)
    print("Baris duplikat:", int(df.duplicated().sum()))
    miss = df.isna().sum()
    print("Missing value per kolom:", miss[miss > 0].to_dict() or "tidak ada")
    print("Baris dengan >=1 missing:", int(df.isna().any(axis=1).sum()))
    print("Distribusi target:", df[kolom_target].value_counts().to_dict())
    print(df.dtypes.value_counts().to_dict())

periksa(raw_hcv, "UCI HCV", "Category")
periksa(raw_hep, "UCI Hepatitis", "Class")

UCI HCV | dimensi: (615, 14)
Baris duplikat: 0
Missing value per kolom: {'ALB': 1, 'ALP': 18, 'ALT': 1, 'CHOL': 10, 'PROT': 1}
Baris dengan >=1 missing: 26
Distribusi target: {'0=Blood Donor': 533, '3=Cirrhosis': 30, '1=Hepatitis': 24, '2=Fibrosis': 21, '0s=suspect Blood Donor': 7}
{dtype('float64'): 10, dtype('int64'): 2, dtype('O'): 2}
UCI Hepatitis | dimensi: (155, 20)
Baris duplikat: 0
Missing value per kolom: {'STEROID': 1, 'FATIGUE': 1, 'MALAISE': 1, 'ANOREXIA': 1, 'LIVER_BIG': 10, 'LIVER_FIRM': 11, 'SPLEEN_PALPABLE': 5, 'SPIDERS': 5, 'ASCITES': 5, 'VARICES': 5, 'BILIRUBIN': 6, 'ALK_PHOSPHATE': 29, 'SGOT': 4, 'ALBUMIN': 16, 'PROTIME': 67}
Baris dengan >=1 missing: 75
Distribusi target: {2: 123, 1: 32}
{dtype('float64'): 15, dtype('int64'): 5}


In [5]:
def siapkan_hcv(df):
    y = (~df["Category"].str.startswith("0")).astype(int)          # 0=Blood Donor & 0s=suspect -> 0
    X = df.drop(columns=["X", "Category"]).copy()
    X["Sex"] = (X["Sex"] == "m").astype(int)
    return X.reset_index(drop=True), y.reset_index(drop=True)

def siapkan_hep(df):
    y = (df["Class"] == 1).astype(int)                             # 1 = DIE = positif
    X = df.drop(columns=["Class"]).copy()
    return X.reset_index(drop=True), y.reset_index(drop=True)

DATASETS = {"HCV": siapkan_hcv(raw_hcv), "Hepatitis": siapkan_hep(raw_hep)}

baris = []
for nama, (X, y) in DATASETS.items():
    maj = max(y.mean(), 1 - y.mean())
    baris.append({"dataset": nama, "n": len(y), "fitur": X.shape[1], "positif": int(y.sum()),
                  "negatif": int((1 - y).sum()), "persen_positif": round(100 * y.mean(), 1),
                  "baseline_akurasi_mayoritas": round(100 * maj, 1)})
pd.DataFrame(baris)

,dataset,n,fitur,positif,negatif,persen_positif,baseline_akurasi_mayoritas
0,HCV,615,12,75,540,12.2,87.8
1,Hepatitis,155,19,32,123,20.6,79.4


In [6]:
def buat_cv(seed=SEED):
    return RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=seed)

print("Jumlah pembagian per model:", N_SPLITS * N_REPEATS)

Jumlah pembagian per model: 50


In [7]:
class XGBSeimbang(XGBClassifier):
    # class weight untuk XGBoost: rasio negatif/positif dihitung dari data latih lipatan saja
    def fit(self, X, y, **kw):
        y_arr = np.asarray(y)
        self.set_params(scale_pos_weight=(y_arr == 0).sum() / max((y_arr == 1).sum(), 1))
        return super().fit(X, y, **kw)

def klasifier(nama, balance, seed):
    cw = "balanced" if balance == "class_weight" else None
    if nama == "NB":
        return GaussianNB()
    if nama == "DT":
        return DecisionTreeClassifier(class_weight=cw, random_state=seed)
    if nama == "KNN":
        return KNeighborsClassifier()
    if nama == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, class_weight=cw, random_state=seed, n_jobs=1)
    if nama == "GB":
        return GradientBoostingClassifier(n_estimators=N_TREES, random_state=seed)
    if nama == "XGB":
        kelas = XGBSeimbang if balance == "class_weight" else XGBClassifier
        return kelas(n_estimators=N_TREES, eval_metric="logloss", random_state=seed, n_jobs=1, verbosity=0)
    raise ValueError(nama)

GRIDS = {
    "NB":  {"clf__var_smoothing": [1e-9, 1e-7, 1e-5]},
    "DT":  {"clf__max_depth": [3, 5, None], "clf__min_samples_leaf": [1, 5]},
    "KNN": {"clf__n_neighbors": [3, 5, 9]},
    "RF":  {"clf__max_depth": [None, 5], "clf__min_samples_leaf": [1, 3]},
    "GB":  {"clf__max_depth": [2, 3], "clf__learning_rate": [0.05, 0.2]},
    "XGB": {"clf__max_depth": [2, 4], "clf__learning_rate": [0.05, 0.2]},
}
BERLAKU = {"none": set(MODELS), "class_weight": {"DT", "RF", "XGB"}, "smote": set(MODELS)}

def buat_estimator(nama, balance, imputer="median", use_fs=True, n_fitur=None, tune=True, seed=SEED):
    # imputer: "median", "knn", atau "none" (data sudah tanpa NaN)
    langkah = []
    if imputer == "median":
        langkah.append(("imp", SimpleImputer(strategy="median")))
    elif imputer == "knn":
        langkah.append(("imp", KNNImputer(n_neighbors=5)))
    if nama == "KNN":
        langkah.append(("scale", StandardScaler()))
    if use_fs:
        langkah.append(("fs", SelectKBest(f_classif, k=min(K_FEATURES, n_fitur))))
    if balance == "smote":
        langkah.append(("smote", SMOTE(random_state=seed, k_neighbors=3)))
    langkah.append(("clf", klasifier(nama, balance, seed)))
    pipa = Pipeline(langkah)
    if tune:
        return GridSearchCV(pipa, GRIDS[nama], scoring="average_precision",
                            cv=StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed), n_jobs=1,
                            error_score="raise")
    return pipa

def rapikan(res, dataset, model, prosedur, config="utama"):
    d = pd.DataFrame({m: res[f"test_{m}"] for m in METRICS})
    d.insert(0, "split", np.arange(len(d)))
    d.insert(0, "config", config)
    d.insert(0, "prosedur", prosedur)
    d.insert(0, "model", model)
    d.insert(0, "dataset", dataset)
    return d

NAMA_PIPA = {"none": "P-A", "class_weight": "P-B", "smote": "P-C"}

def jalankan_pipa(X, y, dataset, nama, balance, imputer="median", seed=SEED, config="utama"):
    est = buat_estimator(nama, balance, imputer, use_fs=True, n_fitur=X.shape[1], seed=seed)
    res = cross_validate(est, X, y, cv=buat_cv(seed), scoring=SCORING, n_jobs=-1, error_score="raise")
    return rapikan(res, dataset, nama, NAMA_PIPA[balance], config)

In [8]:
def jalankan_semua_pipa(models=MODELS, imputer="median", seed=SEED, config="utama", balances=("none", "class_weight", "smote")):
    hasil = []
    for ds, (X, y) in DATASETS.items():
        for nama in models:
            for b in balances:
                if nama not in BERLAKU[b]:
                    continue
                t0 = time.time()
                hasil.append(jalankan_pipa(X, y, ds, nama, b, imputer, seed, config))
                print(f"[{ds}] {nama:3s} {NAMA_PIPA[b]}  ({time.time() - t0:.0f} dtk)")
    return pd.concat(hasil, ignore_index=True)

In [9]:
def impute_global(X, imputer="median"):
    imp = SimpleImputer(strategy="median") if imputer == "median" else KNNImputer(n_neighbors=5)
    return pd.DataFrame(imp.fit_transform(X), columns=X.columns)       # dipelajari dari seluruh baris (sebelum pembagian)

def smote_global(Xi, y, seed):
    Xr, yr = SMOTE(random_state=seed, k_neighbors=3).fit_resample(Xi, y)   # dilakukan pada seluruh data sebelum pembagian
    Xr, yr = np.asarray(Xr), np.asarray(yr)
    n = len(y)
    # imblearn menaruh baris asli di urutan awal, titik sintetis di belakang. Dicek agar penandaan baris asli pasti benar.
    assert np.allclose(Xr[:n], np.asarray(Xi, dtype=float)) and (yr[:n] == np.asarray(y)).all(), \
        "Urutan keluaran SMOTE tidak sesuai: baris asli tidak berada di awal."
    assert (yr[n:] == 1).all(), "Titik sintetis seharusnya semuanya kelas positif."
    return Xr, yr

SCORERS = {m: (get_scorer(s) if isinstance(s, str) else s) for m, s in SCORING.items()}

def _skor(est, X, y):
    return {f"test_{m}": SCORERS[m](est, X, y) for m in METRICS}

def _satu_lipatan_b(est, X_latih, y_latih, X_uji, y_uji):
    e = clone(est).fit(X_latih, y_latih)
    return _skor(e, X_uji, y_uji)

def cv_tambah_sintetis(est, Xi, y, X_sintetis, cv, kolom=None):
    """Pembagian dibuat dari baris asli (sama dengan P-C). Semua titik sintetis ditambahkan ke data latih;
    data uji hanya berisi baris asli. kolom: indeks fitur terpilih (untuk V4b)."""
    Xi = np.asarray(Xi, dtype=float); y = np.asarray(y)
    if kolom is not None:
        Xi, X_sintetis = Xi[:, kolom], X_sintetis[:, kolom]
    y_sint = np.ones(len(X_sintetis), dtype=int)
    tugas = (delayed(_satu_lipatan_b)(est, np.vstack([Xi[tr], X_sintetis]), np.concatenate([y[tr], y_sint]),
                                      Xi[te], y[te]) for tr, te in cv.split(Xi, y))
    hasil = Parallel(n_jobs=-1)(tugas)
    return {k: np.array([h[k] for h in hasil]) for k in hasil[0]}

def jalankan_variasi(X, y, dataset, nama, skenario, imputer="median", seed=SEED, config="utama"):
    cv = buat_cv(seed)
    n = len(y)
    if skenario == "V1":
        Xi = impute_global(X, imputer)
        est = buat_estimator(nama, "smote", "none", use_fs=True, n_fitur=Xi.shape[1], seed=seed)
        res = cross_validate(est, Xi, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario == "V2":
        Xr, yr = smote_global(impute_global(X, imputer), y, seed)
        est = buat_estimator(nama, "none", "none", use_fs=True, n_fitur=Xr.shape[1], seed=seed)
        res = cross_validate(est, Xr, yr, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario == "V2b":
        Xi = impute_global(X, imputer)
        Xr, yr = smote_global(Xi, y, seed)
        est = buat_estimator(nama, "none", "none", use_fs=True, n_fitur=Xi.shape[1], seed=seed)
        res = cv_tambah_sintetis(est, Xi, y, Xr[n:], cv)
    elif skenario == "V3":
        Xi = impute_global(X, imputer)
        terpilih = SelectKBest(f_classif, k=min(K_FEATURES, Xi.shape[1])).fit(Xi, y).get_support()   # fitur dipilih dari seluruh data
        Xs = X.loc[:, terpilih]
        est = buat_estimator(nama, "smote", imputer, use_fs=False, seed=seed)
        res = cross_validate(est, Xs, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario in ("V4", "V4b"):
        Xi = impute_global(X, imputer)
        Xr, yr = smote_global(Xi, y, seed)
        terpilih = SelectKBest(f_classif, k=min(K_FEATURES, Xr.shape[1])).fit(Xr, yr).get_support()  # fitur dipilih dari seluruh data
        kolom = np.flatnonzero(terpilih)
        terbaik = None
        for params in ParameterGrid(GRIDS[nama]):               # hiperparameter dipilih berdasarkan skor data uji
            est = buat_estimator(nama, "none", "none", use_fs=False, tune=False, seed=seed).set_params(**params)
            if skenario == "V4":
                r = cross_validate(est, Xr[:, kolom], yr, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
            else:
                r = cv_tambah_sintetis(est, Xi, y, Xr[n:], cv, kolom=kolom)
            skor = r["test_average_precision"].mean()
            if terbaik is None or skor > terbaik[0]:
                terbaik = (skor, r)
        res = terbaik[1]
    else:
        raise ValueError(skenario)
    return rapikan(res, dataset, nama, skenario, config)

SEMUA_VARIASI = ("V1", "V2", "V2b", "V3", "V4", "V4b")

def jalankan_semua_variasi(models=MODELS, skenario=SEMUA_VARIASI, imputer="median", seed=SEED, config="utama"):
    hasil = []
    for ds, (X, y) in DATASETS.items():
        for nama in models:
            for s in skenario:
                t0 = time.time()
                hasil.append(jalankan_variasi(X, y, ds, nama, s, imputer, seed, config))
                print(f"[{ds}] {nama:3s} {s:3s}  ({time.time() - t0:.0f} dtk)")
    return pd.concat(hasil, ignore_index=True)

In [10]:
def ttest_terkoreksi(d, n_splits=N_SPLITS):
    """Corrected resampled t-test (Nadeau & Bengio 2003) untuk selisih per lipatan dari repeated k-fold CV."""
    d = np.asarray(d, dtype=float)
    J = len(d)
    rasio = 1.0 / (n_splits - 1)                     # n_uji / n_latih
    rata, var = d.mean(), d.var(ddof=1)
    se = np.sqrt((1.0 / J + rasio) * var)
    tkrit = stats.t.ppf(0.975, J - 1)
    if se == 0:                                      # semua selisih identik
        return rata, rata, rata, np.nan, np.nan
    t = rata / se
    p = 2 * stats.t.sf(abs(t), J - 1)
    return rata, rata - tkrit * se, rata + tkrit * se, t, p

def holm(p):
    p = np.asarray(p, dtype=float); out = np.full_like(p, np.nan)
    ok = ~np.isnan(p); q = p[ok]; m = len(q)
    if m == 0: return out
    idx = np.argsort(q)
    adj = np.maximum.accumulate((m - np.arange(m)) * q[idx])
    res = np.empty(m); res[idx] = np.minimum(adj, 1.0)
    out[ok] = res
    return out

def bh(p):
    p = np.asarray(p, dtype=float); out = np.full_like(p, np.nan)
    ok = ~np.isnan(p); q = p[ok]; m = len(q)
    if m == 0: return out
    idx = np.argsort(q)
    adj = q[idx] * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    res = np.empty(m); res[idx] = np.minimum(adj, 1.0)
    out[ok] = res
    return out

BERPASANGAN = ("V1", "V2b", "V3", "V4b")

def analisis_selisih(df, ref="P-C", metrik=("accuracy", "balanced_accuracy", "recall", "f1", "average_precision", "roc_auc"),
                     skenario=SEMUA_VARIASI, alpha=0.05):
    baris = []
    for (ds, model, cfg), g in df.groupby(["dataset", "model", "config"]):
        c = g[g.prosedur == ref].sort_values("split")
        if c.empty:
            continue
        for k in skenario:
            w = g[g.prosedur == k].sort_values("split")
            if w.empty:
                continue
            for m in metrik:
                a, b = c[m].to_numpy(), w[m].to_numpy()
                if k in BERPASANGAN:
                    assert len(a) == len(b), "Jumlah lipatan berbeda; pembagian tidak sejajar."
                    est, lo, hi, t, p = ttest_terkoreksi(b - a)
                    uji = "corrected resampled t-test (berpasangan)"
                else:
                    est, lo, hi, t, p = b.mean() - a.mean(), np.nan, np.nan, np.nan, np.nan
                    uji = "deskriptif (tidak berpasangan)"
                baris.append({"dataset": ds, "model": model, "config": cfg, "skenario": k, "metrik": m,
                              "selisih_rata2": est, "ci95_low": lo, "ci95_high": hi, "t": t,
                              "p_value": p, "uji": uji})
    out = pd.DataFrame(baris)
    # koreksi uji ganda: per konfigurasi dan per metrik
    out["p_holm"] = np.nan; out["p_bh"] = np.nan
    for _, idx in out.groupby(["config", "metrik"]).groups.items():
        out.loc[idx, "p_holm"] = holm(out.loc[idx, "p_value"])
        out.loc[idx, "p_bh"] = bh(out.loc[idx, "p_value"])
    def label(r):
        if np.isnan(r.p_value):
            return "deskriptif" if r.uji.startswith("deskriptif") else "tidak berbeda"
        if r.p_holm < alpha:
            return "lebih tinggi" if r.selisih_rata2 > 0 else "lebih rendah"
        return "tidak berbeda"
    out["kesimpulan_holm"] = out.apply(label, axis=1)
    return out

In [11]:
UTAMA = pd.read_csv(os.path.join(OUT_DIR, "hasil_semua_mentah.csv"))   # hasil eksperimen utama
print(UTAMA.shape, sorted(UTAMA.prosedur.unique()))

(5100, 12) ['P-A', 'P-B', 'P-C', 'V1', 'V2', 'V2b', 'V3', 'V4', 'V4b']


## Bagian A. Duplikat dan jenis fitur (komentar *Duplicates* dan *Imputation of categorical features*)

In [12]:
# A1. Duplikat: baris identik penuh, tanpa ID, dan tanpa ID + label
dup = []
for nama, raw, id_kol, target in [("HCV", raw_hcv, ["X"], "Category"), ("Hepatitis", raw_hep, [], "Class")]:
    dup.append({"dataset": nama, "n": len(raw),
                "duplikat_baris_penuh": int(raw.duplicated().sum()),
                "duplikat_tanpa_ID": int(raw.drop(columns=id_kol).duplicated().sum()),
                "duplikat_fitur_saja (tanpa ID dan label)": int(raw.drop(columns=id_kol + [target]).duplicated().sum())})
dup = pd.DataFrame(dup); dup.to_csv(os.path.join(OUT_DIR, "revisi_A1_duplikat.csv"), index=False)
dup

,dataset,n,duplikat_baris_penuh,duplikat_tanpa_ID,duplikat_fitur_saja (tanpa ID dan label)
0,HCV,615,0,0,0
1,Hepatitis,155,0,0,0


In [13]:
# A2. Jenis fitur UCI Hepatitis dan perilaku imputasi median pada fitur biner (1/2)
Xh, yh = DATASETS["Hepatitis"]
biner = [c for c in Xh if set(Xh[c].dropna().unique()) <= {1, 2}]
numerik = [c for c in Xh if c not in biner]
baris = []
for c in Xh:
    s = Xh[c]
    baris.append({"fitur": c, "jenis": "biner (1/2)" if c in biner else "numerik", "missing": int(s.isna().sum()),
                  "median_seluruh_data": s.median(), "modus_seluruh_data": s.mode()[0],
                  "proporsi_nilai_2": round((s == 2).sum() / s.notna().sum(), 3) if c in biner else np.nan})
jenis = pd.DataFrame(baris)

# Pada setiap lipatan latih: apakah median fitur biner sama dengan modus? Median 1,5 hanya terjadi bila 1 dan 2 tepat seimbang.
cek = []
for seed in (SEED, 7, 2024):
    for i, (tr, te) in enumerate(RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=seed).split(Xh, yh)):
        for c in biner:
            if not Xh[c].isna().any():
                continue
            s = Xh[c].iloc[tr]
            med = s.median()
            sel_imputasi = int(Xh[c].isna().sum())   # baris yang akan diisi (latih + uji)
            cek.append({"seed": seed, "lipatan": i, "fitur": c, "median_latih": med,
                        "median_bukan_1_atau_2": med not in (1.0, 2.0)})
cek = pd.DataFrame(cek)
ringkas_cek = cek.groupby(["seed", "fitur"])["median_bukan_1_atau_2"].sum().unstack("seed")
jenis.to_csv(os.path.join(OUT_DIR, "revisi_A2_jenis_fitur.csv"), index=False)
ringkas_cek.to_csv(os.path.join(OUT_DIR, "revisi_A2_median_biner_per_lipatan.csv"))
print("Fitur biner:", len(biner), biner)
print("Fitur numerik:", len(numerik), numerik)
print("\nJumlah lipatan (dari 50) dengan median latih = 1,5 (seri), per fitur dan seed:")
display(ringkas_cek[ringkas_cek.sum(axis=1) > 0])
jenis

Fitur biner: 13 ['SEX', 'STEROID', 'ANTIVIRALS', 'FATIGUE', 'MALAISE', 'ANOREXIA', 'LIVER_BIG', 'LIVER_FIRM', 'SPLEEN_PALPABLE', 'SPIDERS', 'ASCITES', 'VARICES', 'HISTOLOGY']
Fitur numerik: 6 ['AGE', 'BILIRUBIN', 'ALK_PHOSPHATE', 'SGOT', 'ALBUMIN', 'PROTIME']

Jumlah lipatan (dari 50) dengan median latih = 1,5 (seri), per fitur dan seed:


seed,7,42,2024
fitur,,,
STEROID,2,1,2


,fitur,jenis,missing,median_seluruh_data,modus_seluruh_data,proporsi_nilai_2
0,AGE,numerik,0,39.0,30.0,NaN
1,SEX,biner (1/2),0,1.0,1.0,0.103
2,STEROID,biner (1/2),1,2.0,2.0,0.506
3,ANTIVIRALS,biner (1/2),0,2.0,2.0,0.845
4,FATIGUE,biner (1/2),1,1.0,1.0,0.351
5,MALAISE,biner (1/2),1,2.0,2.0,0.604
6,ANOREXIA,biner (1/2),1,2.0,2.0,0.792
7,LIVER_BIG,biner (1/2),10,2.0,2.0,0.828
8,LIVER_FIRM,biner (1/2),11,2.0,2.0,0.583
9,SPLEEN_PALPABLE,biner (1/2),5,2.0,2.0,0.800


## Bagian B. Bukti gabungan untuk kebocoran sungguhan (komentar *Pooled evidence*)
Tiga ringkasan, semuanya dari skor per lipatan yang sudah ada:
1. **Uji tanda** atas arah selisih V2b − P-C pada 36 kombinasi (2 dataset × 6 model × 3 metrik utama). Kombinasi ini tidak saling bebas (lipatan dan metrik sama), jadi uji tanda dibaca sebagai ringkasan deskriptif.
2. **Corrected resampled t-test gabungan.** Karena keenam model memakai lipatan yang sama, selisih per lipatan dirata-ratakan lintas model, lalu diuji dengan *corrected resampled t-test*. Ini menguji efek rata-rata untuk himpunan model ini, dan memperhitungkan korelasi antarmodel dengan sendirinya.
3. **Bayesian correlated t-test** (Corani & Benavoli, 2015; Benavoli et al., 2017) dengan ROPE ±0,01. Posterior selisih berupa distribusi Student-t dengan df = J − 1, lokasi = rata-rata selisih, dan skala = galat baku terkoreksi (sama dengan uji Nadeau-Bengio). Hasilnya berupa peluang posterior P(selisih > 0,01), P(dalam ROPE), dan P(selisih < −0,01). Uji ini juga bisa menyatakan **kesetaraan praktis** (misalnya untuk V1), yang tidak bisa dinyatakan oleh p-value.

In [14]:
ROPE = 0.01
METRIK_UTAMA = ["f1", "average_precision", "balanced_accuracy"]
METRIK_B = METRIK_UTAMA + ["roc_auc"]

def selisih_per_lipatan(df, varian, ref="P-C"):
    a = df[df.prosedur == ref].set_index(["dataset", "model", "split"])[METRIK_B]
    b = df[df.prosedur == varian].set_index(["dataset", "model", "split"])[METRIK_B]
    return (b - a).dropna()

def bayes_corr_ttest(d, n_splits=N_SPLITS, rope=ROPE):
    d = np.asarray(d, float); J = len(d)
    se = np.sqrt((1 / J + 1 / (n_splits - 1)) * d.var(ddof=1))
    post = stats.t(J - 1, loc=d.mean(), scale=se)
    return post.sf(rope), post.cdf(rope) - post.cdf(-rope), post.cdf(-rope)

baris_tanda, baris_gab = [], []
for v in ["V1", "V2b", "V3", "V4b"]:
    D = selisih_per_lipatan(UTAMA, v)
    rata_model = D.groupby(level=["dataset", "model"]).mean()
    pos = int((rata_model[METRIK_UTAMA] > 0).values.sum()); n = rata_model[METRIK_UTAMA].size
    baris_tanda.append({"varian": v, "kombinasi_positif": pos, "kombinasi": n,
                        "p_uji_tanda_dua_sisi": stats.binomtest(pos, n, 0.5).pvalue})
    for ds in ["HCV", "Hepatitis"]:
        gab = D.loc[ds].groupby(level="split").mean()          # rata-rata lintas 6 model per lipatan
        for m in METRIK_B:
            rata, lo, hi, t, p = ttest_terkoreksi(gab[m].values)
            pg, pr, pl = bayes_corr_ttest(gab[m].values)
            baris_gab.append({"varian": v, "dataset": ds, "metrik": m, "selisih_gabungan": rata,
                              "ci95_low": lo, "ci95_high": hi, "p_terkoreksi": p,
                              "P(selisih>ROPE)": pg, "P(dalam ROPE)": pr, "P(selisih<-ROPE)": pl,
                              "model_positif": f"{int((rata_model.loc[ds][m] > 0).sum())}/6"})
tanda = pd.DataFrame(baris_tanda)
gabung = pd.DataFrame(baris_gab)
gabung["p_holm"] = np.nan
for _, idx in gabung.groupby("metrik").groups.items():       # Holm per metrik (2 dataset x 4 varian)
    gabung.loc[idx, "p_holm"] = holm(gabung.loc[idx, "p_terkoreksi"])
tanda.to_csv(os.path.join(OUT_DIR, "revisi_B_uji_tanda.csv"), index=False)
gabung.to_csv(os.path.join(OUT_DIR, "revisi_B_bukti_gabungan.csv"), index=False)
display(tanda)
gabung.round(4)

,varian,kombinasi_positif,kombinasi,p_uji_tanda_dua_sisi
0,V1,24,36,6.524534e-02
1,V2b,35,36,1.076842e-09
2,V3,33,36,2.272136e-07
3,V4b,34,36,1.941225e-08


,varian,dataset,metrik,selisih_gabungan,ci95_low,ci95_high,p_terkoreksi,P(selisih>ROPE),P(dalam ROPE),P(selisih<-ROPE),model_positif,p_holm
0,V1,HCV,f1,0.0010,-0.0083,0.0104,0.8247,0.0299,0.9593,0.0108,5/6,1.0000
1,V1,HCV,average_precision,0.0020,-0.0073,0.0113,0.6675,0.0448,0.9490,0.0062,3/6,1.0000
2,V1,HCV,balanced_accuracy,0.0014,-0.0045,0.0072,0.6367,0.0024,0.9975,0.0001,6/6,1.0000
3,V1,HCV,roc_auc,0.0005,-0.0034,0.0043,0.8144,0.0000,1.0000,0.0000,3/6,1.0000
4,V1,Hepatitis,f1,0.0007,-0.0259,0.0272,0.9596,0.2420,0.5462,0.2118,4/6,1.0000
5,V1,Hepatitis,average_precision,-0.0018,-0.0189,0.0153,0.8310,0.0856,0.7433,0.1711,2/6,1.0000
6,V1,Hepatitis,balanced_accuracy,0.0004,-0.0162,0.0170,0.9605,0.1256,0.7678,0.1067,4/6,1.0000
7,V1,Hepatitis,roc_auc,0.0016,-0.0098,0.0131,0.7770,0.0742,0.9022,0.0235,3/6,1.0000
8,V2b,HCV,f1,0.0678,0.0078,0.1278,0.0276,0.9707,0.0233,0.0061,6/6,0.1996
9,V2b,HCV,average_precision,0.0726,0.0249,0.1203,0.0036,0.9944,0.0051,0.0005,6/6,0.0252


## Bagian C. Seleksi fitur dengan k lebih kecil (komentar *Feature selection with fixed k = 8*)
P-C dan V3 dijalankan ulang dengan k = 3 dan k = 5 (UCI HCV memiliki 12 fitur, UCI Hepatitis 19). Keduanya memakai k yang sama, sehingga perbedaannya hanya data yang dipakai untuk memilih fitur. Hasil k = 8 diambil dari eksperimen utama.

In [15]:
K_ASLI = K_FEATURES
hasil_k = []
for k in (3, 5):
    K_FEATURES = k                                   # dipakai oleh buat_estimator dan jalankan_variasi
    cfg = f"k={k}"
    print("=== k =", k)
    hasil_k.append(jalankan_semua_pipa(balances=("smote",), config=cfg))
    hasil_k.append(jalankan_semua_variasi(skenario=("V3",), config=cfg))
K_FEATURES = K_ASLI
hasil_k = pd.concat(hasil_k + [UTAMA[UTAMA.prosedur.isin(["P-C", "V3"]) & UTAMA.model.isin(MODELS)].assign(config="k=8")], ignore_index=True)
hasil_k.to_csv(os.path.join(OUT_DIR, "revisi_C_k_mentah.csv"), index=False)

sel_k = analisis_selisih(hasil_k, skenario=("V3",))
sel_k.to_csv(os.path.join(OUT_DIR, "revisi_C_selisih_V3_per_k.csv"), index=False)
tabel_k = (sel_k[sel_k.metrik.isin(["f1", "average_precision", "balanced_accuracy"])]
           .groupby(["dataset", "metrik", "config"])["selisih_rata2"].mean().unstack("config").round(3))
nyata_k = (sel_k[sel_k.metrik == "f1"].assign(naik=lambda d: d.kesimpulan_holm == "lebih tinggi")
           .groupby(["dataset", "config"])["naik"].sum().unstack("config"))
pc_k = hasil_k[hasil_k.prosedur == "P-C"].groupby(["dataset", "config"])["f1"].mean().unstack("config").round(3)
print("Rata-rata selisih V3 - P-C lintas model, per k:"); display(tabel_k)
print("Jumlah model dengan F1 V3 lebih tinggi nyata (Holm), per k:"); display(nyata_k)
print("Rata-rata F1 P-C lintas model, per k:"); pc_k

=== k = 3
[HCV] NB  P-C  (9 dtk)
[HCV] DT  P-C  (7 dtk)
[HCV] KNN P-C  (4 dtk)
[HCV] RF  P-C  (107 dtk)
[HCV] GB  P-C  (65 dtk)
[HCV] XGB P-C  (17 dtk)
[Hepatitis] NB  P-C  (4 dtk)
[Hepatitis] DT  P-C  (7 dtk)
[Hepatitis] KNN P-C  (4 dtk)
[Hepatitis] RF  P-C  (101 dtk)
[Hepatitis] GB  P-C  (55 dtk)
[Hepatitis] XGB P-C  (13 dtk)
[HCV] NB  V3   (5 dtk)
[HCV] DT  V3   (6 dtk)
[HCV] KNN V3   (4 dtk)
[HCV] RF  V3   (108 dtk)
[HCV] GB  V3   (75 dtk)
[HCV] XGB V3   (17 dtk)
[Hepatitis] NB  V3   (4 dtk)
[Hepatitis] DT  V3   (6 dtk)
[Hepatitis] KNN V3   (4 dtk)
[Hepatitis] RF  V3   (99 dtk)
[Hepatitis] GB  V3   (50 dtk)
[Hepatitis] XGB V3   (12 dtk)
=== k = 5
[HCV] NB  P-C  (4 dtk)
[HCV] DT  P-C  (7 dtk)
[HCV] KNN P-C  (5 dtk)
[HCV] RF  P-C  (117 dtk)
[HCV] GB  P-C  (84 dtk)
[HCV] XGB P-C  (16 dtk)
[Hepatitis] NB  P-C  (4 dtk)
[Hepatitis] DT  P-C  (7 dtk)
[Hepatitis] KNN P-C  (4 dtk)
[Hepatitis] RF  P-C  (97 dtk)
[Hepatitis] GB  P-C  (52 dtk)
[Hepatitis] XGB P-C  (12 dtk)
[HCV] NB  V3   (3 dtk)

config                         k=3    k=5    k=8
dataset   metrik                                
HCV       average_precision  0.000  0.000  0.021
          balanced_accuracy  0.000 -0.001  0.018
          f1                 0.000  0.001  0.028
Hepatitis average_precision  0.030  0.005  0.012
          balanced_accuracy  0.008  0.002  0.007
          f1                 0.014  0.002  0.010

Jumlah model dengan F1 V3 lebih tinggi nyata (Holm), per k:


config,k=3,k=5,k=8
dataset,,,
HCV,0,0,0
Hepatitis,0,0,0


Rata-rata F1 P-C lintas model, per k:


config,k=3,k=5,k=8
dataset,,,
HCV,0.667,0.693,0.753
Hepatitis,0.483,0.504,0.554


## Bagian D. V2b dengan 80% titik sintetis (komentar *Decomposition caveat*)
Pada V2, sekitar 80% titik sintetis masuk data latih (sisanya jatuh ke data uji), sedangkan V2b memakai 100%. **V2b-80** sama dengan V2b, tetapi pada setiap lipatan hanya 80% titik sintetis yang diambil secara acak (seed tetap per lipatan) untuk ditambahkan ke data latih. Jika V2b-80 ≈ V2b, perbedaan jumlah titik sintetis tidak memengaruhi dekomposisi.

In [16]:
FRAKSI_SINTETIS = 0.8

def _lipatan_sub(est, Xi, y, X_sint, tr, te, fraksi, seed_lipatan):
    rng = np.random.default_rng(seed_lipatan)
    pilih = rng.choice(len(X_sint), size=int(round(fraksi * len(X_sint))), replace=False)
    Xs = X_sint[pilih]
    return _satu_lipatan_b(est, np.vstack([Xi[tr], Xs]), np.concatenate([y[tr], np.ones(len(Xs), int)]), Xi[te], y[te])

def jalankan_v2b_sub(X, y, dataset, nama, fraksi=FRAKSI_SINTETIS, imputer="median", seed=SEED):
    Xi = impute_global(X, imputer)
    Xr, yr = smote_global(Xi, y, seed)
    est = buat_estimator(nama, "none", "none", use_fs=True, n_fitur=Xi.shape[1], seed=seed)
    Xa, ya = np.asarray(Xi, float), np.asarray(y)
    tugas = (delayed(_lipatan_sub)(est, Xa, ya, Xr[len(y):], tr, te, fraksi, seed * 1000 + i)
             for i, (tr, te) in enumerate(buat_cv(seed).split(Xa, ya)))
    hasil = Parallel(n_jobs=-1)(tugas)
    res = {k: np.array([h[k] for h in hasil]) for k in hasil[0]}
    return rapikan(res, dataset, nama, f"V2b-{int(fraksi * 100)}")

hasil_d = []
for ds, (X, y) in DATASETS.items():
    for nama in MODELS:
        t0 = time.time()
        hasil_d.append(jalankan_v2b_sub(X, y, ds, nama))
        print(f"[{ds}] {nama:3s} V2b-80  ({time.time() - t0:.0f} dtk)")
hasil_d = pd.concat(hasil_d, ignore_index=True)
hasil_d.to_csv(os.path.join(OUT_DIR, "revisi_D_v2b80_mentah.csv"), index=False)

gab_d = pd.concat([UTAMA[UTAMA.prosedur.isin(["P-C", "V2", "V2b"]) & UTAMA.model.isin(MODELS)], hasil_d], ignore_index=True)
sel_d = analisis_selisih(gab_d.assign(config="utama"), skenario=("V2", "V2b"))
BERPASANGAN_ASLI = BERPASANGAN
BERPASANGAN = BERPASANGAN + ("V2b-80",)
sel_d80 = analisis_selisih(gab_d.assign(config="utama"), skenario=("V2b-80",))
BERPASANGAN = BERPASANGAN_ASLI
sel_d = pd.concat([sel_d, sel_d80], ignore_index=True)
sel_d.to_csv(os.path.join(OUT_DIR, "revisi_D_selisih.csv"), index=False)

m4 = ["f1", "average_precision", "balanced_accuracy", "roc_auc"]
piv = (sel_d[sel_d.metrik.isin(m4)].groupby(["dataset", "metrik", "skenario"])["selisih_rata2"].mean()
       .unstack("skenario"))
piv["porsi_kebocoran_V2b_%"] = 100 * piv["V2b"] / piv["V2"]
piv["porsi_kebocoran_V2b-80_%"] = 100 * piv["V2b-80"] / piv["V2"]
piv["selisih_V2b-80_vs_V2b"] = piv["V2b-80"] - piv["V2b"]
piv.round(3).to_csv(os.path.join(OUT_DIR, "revisi_D_dekomposisi_80_vs_100.csv"))

# Selisih berpasangan V2b-80 vs V2b per model (lipatan sama)
a = UTAMA[UTAMA.prosedur == "V2b"].set_index(["dataset", "model", "split"])[m4]
b = hasil_d.set_index(["dataset", "model", "split"])[m4]
per_model = (b - a).dropna().groupby(level=["dataset", "model"]).mean().round(3)
per_model.to_csv(os.path.join(OUT_DIR, "revisi_D_v2b80_minus_v2b_per_model.csv"))
print("Dekomposisi dengan 100% vs 80% titik sintetis (rata-rata lintas model):"); display(piv.round(3))
print("Selisih V2b-80 dikurangi V2b per model:"); per_model

[HCV] NB  V2b-80  (2 dtk)
[HCV] DT  V2b-80  (4 dtk)
[HCV] KNN V2b-80  (4 dtk)
[HCV] RF  V2b-80  (115 dtk)
[HCV] GB  V2b-80  (89 dtk)
[HCV] XGB V2b-80  (17 dtk)
[Hepatitis] NB  V2b-80  (2 dtk)
[Hepatitis] DT  V2b-80  (3 dtk)
[Hepatitis] KNN V2b-80  (3 dtk)
[Hepatitis] RF  V2b-80  (98 dtk)
[Hepatitis] GB  V2b-80  (51 dtk)
[Hepatitis] XGB V2b-80  (10 dtk)
Dekomposisi dengan 100% vs 80% titik sintetis (rata-rata lintas model):


skenario                        V2    V2b  V2b-80  porsi_kebocoran_V2b_%  porsi_kebocoran_V2b-80_%  selisih_V2b-80_vs_V2b
dataset   metrik                                                                                                         
HCV       average_precision  0.165  0.073   0.078                 44.070                    47.433                  0.006
          balanced_accuracy  0.061  0.058   0.058                 95.287                    94.986                 -0.000
          f1                 0.183  0.068   0.077                 36.953                    42.040                  0.009
          roc_auc            0.037  0.033   0.034                 89.395                    91.491                  0.001
Hepatitis average_precision  0.303  0.060   0.045                 19.755                    14.928                 -0.015
          balanced_accuracy  0.115  0.044   0.037                 38.286                    32.211                 -0.007
          f1                 0.298  0.056   0.049                 18.716                    16.475                 -0.007
          roc_auc            0.101  0.048   0.038                 47.138                    37.836                 -0.009

Selisih V2b-80 dikurangi V2b per model:


f1  average_precision  balanced_accuracy  roc_auc
dataset   model                                                      
HCV       DT     0.004              0.027             -0.003    0.003
          GB     0.011             -0.000              0.001   -0.000
          KNN    0.023              0.007              0.003    0.001
          NB    -0.004             -0.000             -0.005    0.001
          RF     0.011             -0.001              0.001   -0.000
          XGB    0.012              0.000              0.001   -0.000
Hepatitis DT    -0.017             -0.030             -0.014   -0.022
          GB    -0.023             -0.033             -0.017   -0.015
          KNN    0.022              0.003              0.016   -0.009
          NB     0.003             -0.003             -0.000   -0.000
          RF    -0.005             -0.014             -0.008   -0.006
          XGB   -0.021             -0.012             -0.018   -0.004

In [17]:
# Simpan versi paket untuk addendum
import json
from importlib.metadata import version, PackageNotFoundError
v = {}
for p in ["numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn", "xgboost"]:
    try: v[p] = version(p)
    except PackageNotFoundError: v[p] = "tidak ditemukan"
info = {"pelaksana": NAMA_PELAKSANA, "python": platform.python_version(), "tanggal_jalan": str(datetime.date.today()), "SEED": SEED,
        "N_REPEATS": N_REPEATS, "FAST_MODE": FAST_MODE, "versi_paket": v}
json.dump(info, open(os.path.join(OUT_DIR, "revisi_konfigurasi.json"), "w"), indent=2)
info

{'pelaksana': 'Ade',
 'python': '3.11.8',
 'tanggal_jalan': '2026-11-05',
 'SEED': 42,
 'N_REPEATS': 10,
 'FAST_MODE': False,
 'versi_paket': {'numpy': '1.26.4',
  'pandas': '2.2.1',
  'scipy': '1.12.0',
  'scikit-learn': '1.9.1',
  'imbalanced-learn': '0.14.2',
  'xgboost': '2.1.4'}}